# 설정 관리와 텐서보드 사용 



- 파이터치에서의 config, 즉 설정은 모델 학습, 평가, 추론 등을 수행할 때 필요한 다양한 매개변수와 환경을 정의하는 것 
- 이설정은 모델의 아키텍처, 학습률, 배치크기, 에포크 수, 데이터셋 경로, 최적화 방법 등을 포함할 수 있으며, 실험의 재현성과 코드의 유연성을 높이기 위해 중요 

## config의 중요성

- 재현성: 동일한 설정을 사용함으로써 실험 결과를 재현, 이는 연구 과정에서 매우 중요, 다른 연구자가 당신의 실험을 검증하거나 기반으로 새운 연구를 진행할 수 있게 함. 
- 유연성: 다양한 실험 조건을 쉽게 변경하고 테스트 할 수 있다. 학습률이나 배치크기를 조정하여 모델 성능에 미치는 영향을 실험
- 모듈성: 모델, 데이터로더, 학습 및 평가루틴 등 다양한 구성 요소의 설정을 분리, 코드를 더 모듈화하고 관리하기쉽게 만듬. 

## 텐서보드 (Tensor Board)

- 기계학습 프로젝트에서 모델 학습의 시각화 및 모니터링을 지원하는 도구 
- 스칼라 추적 : 학습 검증에 걸친 Loss와 정확도(accurary)같은 스칼라 값들의 추적 및 시각화를 지원 
- 모델 그래프 시각화: 텐서보드는 모델의 구조를 그래픽 형태로 시각화, 모델의 아키텍처를 쉽게 이해하고 분석할 수 있게 함
- 이미지, 오디오, 텍스트 데이터 시각화 : 학습 데이터 샘플들을 시각화하여 데이터 세트를 직접적으로 확인 
- 히스토그램 및 분포 시각화: 모델의 weight, 활성화 함수(activation)값들의 분포와 변화를 히스토그램으로 시각화 할 수 있다. 
- 하이퍼파라미터 튜닝: 실험 관리를 위한 도구, 다양한 하이퍼파라미터 설정을 실험하여 최적의 모델 구성을 찾을 수 있도록 지원 

In [3]:
!tensorboard --version

zsh:1: /opt/homebrew/bin/tensorboard: bad interpreter: /opt/homebrew/opt/python@3.10/bin/python3.10: no such file or directory


## tensorboard data correct

### SummaryWritter 객체 

- 텐서보드로 학습 데이터를 수집하고 시각화하기 위해 `torch.utils.tensorboard`모듈을 사용할 수 있다. 

In [4]:
from torch.utils.tensorboard import SummaryWriter

In [5]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from torch.utils.tensorboard import SummaryWriter

# 설정값
config = {
    'input_size': 10,  # 입력 특성의 수
    'output_size': 1,  # 출력 크기
    'batch_size': 64,  # 배치 크기
    'num_epochs': 5,  # 에포크 수
    'learning_rate': 0.01,  # 학습률
}

# 텐서보드 설정
writer = SummaryWriter('runs/simple_linear_model')

# 랜덤 데이터 생성
inputs = torch.randn(1000, config['input_size'])  # 1000개의 샘플, 각 샘플은 input_size 크기의 특성을 가짐
targets = torch.randn(1000, config['output_size'])  # 각 샘플에 대한 랜덤 출력 값

# TensorDataset과 DataLoader 생성
dataset = TensorDataset(inputs, targets)  # TensorDataset으로 랜덤 데이터셋 생성
data_loader = DataLoader(dataset, batch_size=config['batch_size'], shuffle=True)  # DataLoader로 배치 관리 및 셔플

In [6]:
# 간단한 선형 모델 정의
class SimpleLinearModel(nn.Module):
    def __init__(self, input_size, output_size):
        super(SimpleLinearModel, self).__init__()
        self.fc = nn.Linear(input_size, output_size)

    def forward(self, x):
        return self.fc(x)

model = SimpleLinearModel(config['input_size'], config['output_size'])
criterion = nn.MSELoss()  # 회귀 문제를 위한 Mean Squared Error Loss
optimizer = optim.SGD(model.parameters(), lr=config['learning_rate'])

# 모델 구조를 텐서보드에 기록 (더미 입력 사용)
dummy_input = torch.zeros(config['batch_size'], config['input_size'])
writer.add_graph(model, dummy_input)

In [7]:
# 학습 과정
for epoch in range(config['num_epochs']):
    total_loss = 0.0

    for batch_inputs, batch_targets in data_loader:
        # Forward pass
        predictions = model(batch_inputs)
        loss = criterion(predictions, batch_targets)

        # Backward and optimize
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    # 에포크마다 평균 손실 값을 텐서보드에 기록
    avg_loss = total_loss / len(data_loader)
    writer.add_scalar('Training Loss', avg_loss, epoch)

    print(f'Epoch {epoch+1}, Average Loss: {avg_loss}')

# 텐서보드 종료 및 리소스 해제
writer.close()

Epoch 1, Average Loss: 1.1242882274091244
Epoch 2, Average Loss: 1.0228412970900536
Epoch 3, Average Loss: 0.9812654480338097
Epoch 4, Average Loss: 0.9507798440754414
Epoch 5, Average Loss: 0.9355810433626175
